# Particle-trajectory animation -- run_2 (1993-2013), cumulative trails

Parameterised (papermill) notebook that renders the Mercator/Parcels `tracks_v3_*` production run as
an MP4 with **cumulative** trails: every frame's new track segments are painted onto the canvas and
are **never erased**, so the drawn trajectories accumulate for the whole video and nothing ever fades
out.  Only the bright dots at the live particle positions and the date label are redrawn per frame.

Differences from the earlier per-period videos (`trail_len` / `twilight` version):

* **no fading** -- `trail_len` is gone; trails are cumulative (see `alpha`/`linewidth` instead);
* **qualitative colours keyed to the release MONTH** (12 distinct colours + legend), so December and
  January are clearly different -- the old cyclic `twilight` day-of-year map made them identical;
* **wider map** (to the African coast) and a wide 13:6.6 figure that matches that aspect ratio;
* **parallel loading** of the zarr stores (`n_load_procs`), because the whole run is 1534 stores;
* frames are written straight into an **ffmpeg pipe** rather than through `FFMpegWriter`, because
  `FFMpegWriter.grab_frame()` triggers a full figure redraw, which would wipe the accumulated canvas.

## How the cumulative canvas works

1. the map (land, coastlines, gridlines, month legend) is drawn **once**, `fig.canvas.draw()`;
2. `buf = np.asarray(fig.canvas.buffer_rgba())` is a *writable view* of the Agg framebuffer;
3. each frame: the new segments go into a throw-away `LineCollection`, `ax.draw_artist(lc)` blits
   them into `buf`, and the collection is immediately removed from the axes (so it is never drawn
   again -- the pixels stay, the artist does not);
4. `snap = buf.copy()` records the trails-only state;
5. the live-position dots and the date text are `draw_artist`-ed on top, `buf` is piped to ffmpeg,
   and then `buf[:] = snap` restores the trails-only state -- which erases the dots and the date but
   keeps every trail ever drawn.

Nothing in the loop ever calls `fig.canvas.draw()` again, so the cost per frame is proportional to
the *new* segments only and stays flat over the whole video.

In [ ]:
# --- papermill parameters ---
period_start         = "1993-01-06"   # first release date to include
period_end           = "2013-12-28"   # last release date to include
n_particles          = 1000           # randomly chosen trajectories per release
release_spacing_days = 5              # draw a release every N days (stores exist every 5 days)
obs_stride           = 4              # keep every Nth 6-hourly obs at load time, 4 = daily vertices
frame_step_days      = 2              # calendar days per animation frame
trail_alpha          = 0.025          # opacity of ONE trail segment (they accumulate!)
trail_lw             = 0.3            # trail line width (points)
head_size            = 2.5            # marker size of the live-particle dots
fps                  = 6
dpi                  = 120
fig_w, fig_h         = 13.0, 6.6      # inches -> 1560 x 792 px at dpi 120 (both even, needed by x264)
out_dir              = "/work/bk1450/b383184/Amazon/Mercator/notebooks/run_2/animations/video_particles_1993-2013"
out_name             = "particles_1993-2013.mp4"
lon_min, lon_max     = -95.0, 0.0
lat_min, lat_max     = -10.0, 30.0
coast_res            = "50m"
max_frames           = 0              # 0 = all frames, positive N = test render of the first N frames
max_releases         = 0              # 0 = all releases, positive N = test load of the first N
n_load_procs         = 32             # processes used to read the zarr stores
seed                 = 20250101
data_root            = "/work/bk1450/b383184/Amazon/Mercator/notebooks/run_2"
ffmpeg_path          = ""             # empty = auto-detect, imageio_ffmpeg static build has libx264
crf                  = 23
preset               = "medium"

In [ ]:
import os, glob, re, time, gc, subprocess, collections
import numpy as np
import xarray as xr
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
from matplotlib.collections import LineCollection
from matplotlib.patches import Patch
import cartopy.crs as ccrs
import cartopy.feature as cfeature
from multiprocessing import Pool

# ffmpeg: the parcels_3.1.2 env has no ffmpeg on PATH, but imageio_ffmpeg ships a static
# ffmpeg 7.0.2 with libx264.  NOTE: /sw/spack-levante/ffmpeg-4.3.2-rmywep/bin/ffmpeg has NO
# libx264 and must not be used.
if not ffmpeg_path:
    try:
        import imageio_ffmpeg
        ffmpeg_path = imageio_ffmpeg.get_ffmpeg_exe()
    except Exception:
        ffmpeg_path = "/sw/spack-levante/mambaforge-23.11.0-0-Linux-x86_64-rb7hor/bin/ffmpeg"
if not os.path.exists(ffmpeg_path):
    raise RuntimeError("no usable ffmpeg found: " + ffmpeg_path)

os.makedirs(out_dir, exist_ok=True)
out_mp4 = os.path.join(out_dir, out_name)
print("output:", out_mp4)
print("ffmpeg:", ffmpeg_path)

# 12 qualitative colours -- deliberately NOT a hue wheel, so that Dec (black) and Jan (blue) are
# unmistakably different.  tab10 + gold + black.
MONTH_COLORS = ["#1f77b4", "#ff7f0e", "#2ca02c", "#d62728", "#9467bd", "#8c564b",
                "#e377c2", "#7f7f7f", "#bcbd22", "#17becf", "#ffd92f", "#000000"]
MONTH_NAMES  = ["Jan", "Feb", "Mar", "Apr", "May", "Jun",
                "Jul", "Aug", "Sep", "Oct", "Nov", "Dec"]

## 1. Select the release stores

In [ ]:
# The run is split over four seed directories; collect every store and keep the ones whose
# release date falls inside [period_start, period_end].
TRACK_DIRS = ["tracks_v3_11098723", "tracks_v3_45678", "tracks_v3_67891", "tracks_v3_78876"]

pat = re.compile(r"Parcels_run_(\d+)_(\d{4}-\d{2}-\d{2})")
all_stores = []
for d in TRACK_DIRS:
    for p in glob.glob(os.path.join(data_root, d, "Parcels_run_*.zarr")):
        m = pat.search(os.path.basename(p))
        if m:
            all_stores.append((np.datetime64(m.group(2)), p))
all_stores.sort()
print("stores found in total:", len(all_stores))

t0d, t1d = np.datetime64(period_start), np.datetime64(period_end)
in_period = [(d, p) for d, p in all_stores if t0d <= d <= t1d]
print("stores inside period:", len(in_period), in_period[0][0], "..", in_period[-1][0])

selected, last = [], None
for d, p in in_period:
    if last is None or (d - last) / np.timedelta64(1, "D") >= release_spacing_days:
        selected.append((d, p)); last = d
if max_releases:
    selected = selected[:max_releases]
print("releases drawn:", len(selected), "| particles per release:", n_particles,
      "| trajectories drawn in total:", len(selected) * n_particles)

## 2. Load the subsampled trajectories in parallel

Each store is opened by a worker process, `n_particles` trajectory indices are drawn at random
(reproducible: the RNG is seeded from the release date), and `lon`/`lat` are read as `float32` for
those indices and only every `obs_stride`-th obs.  Serial loading of all 1534 stores takes ~13 min;
with `n_load_procs` workers it is a couple of minutes.

Memory: `n_releases x n_particles x (740/obs_stride) x 2 x 4 bytes` ~ 2.3 GB for the full run at
1000 particles and `obs_stride = 4`.

In [ ]:
def _load_one(arg):
    d, p = arg
    ds = xr.open_zarr(p)
    ntraj = ds.sizes["trajectory"]
    rng = np.random.default_rng(seed + int(np.datetime64(d, "D").astype(int)))
    k = min(n_particles, ntraj)
    idx = np.sort(rng.choice(ntraj, size=k, replace=False))
    sub = ds[["lon", "lat"]].isel(trajectory=idx, obs=slice(0, None, obs_stride)).load()
    t0 = ds.time.isel(trajectory=0, obs=0).values.astype("datetime64[ns]")
    nobs_full = int(ds.sizes["obs"])
    ds.close()
    return {"date": d, "t0": t0,
            "lon": sub.lon.values.astype("float32"),
            "lat": sub.lat.values.astype("float32"),
            "nv": int(sub.sizes["obs"]), "nobs_full": nobs_full,
            "month": int(str(d)[5:7])}

t_load = time.time()
with Pool(n_load_procs) as pool:
    releases = pool.map(_load_one, selected, chunksize=4)
releases.sort(key=lambda r: r["date"])
load_s = time.time() - t_load
print("parallel load of %d stores in %.0f s with %d procs (%.3f s/store effective)"
      % (len(releases), load_s, n_load_procs, load_s / max(len(releases), 1)))

print("vertices per release:", dict(collections.Counter(r["nv"] for r in releases)))
print("in-memory size: %.2f GB"
      % (sum(r["lon"].nbytes + r["lat"].nbytes for r in releases) / 1e9))
short = [str(r["date"]) for r in releases if r["nv"] < 25]
if short:
    print("WARNING: truncated releases (<25 vertices):", short)

for r in releases:
    r["color"] = MONTH_COLORS[r["month"] - 1]
gc.collect()

## 3. Frame axis (calendar time)

In [ ]:
day = np.timedelta64(1, "D")                       # one vertex = obs_stride * 6 h = 1 day
assert obs_stride * 6 == 24, "obs_stride must be 4 so that one vertex == one calendar day"

t_first = min(r["t0"] for r in releases)
t_last  = max(r["t0"] + np.timedelta64(int(r["nv"] - 1), "D") for r in releases)
n_all   = int((t_last - t_first) / day) // frame_step_days + 1
n_frames = min(n_all, max_frames) if max_frames else n_all
frame_times = t_first + np.arange(n_frames) * np.timedelta64(frame_step_days, "D")
print("frames:", n_frames, np.datetime_as_string(frame_times[0], unit="D"), "..",
      np.datetime_as_string(frame_times[-1], unit="D"))
print("video duration: %.1f s = %.1f min at %d fps" % (n_frames / fps, n_frames / fps / 60, fps))

## 4. The static map (drawn once)

In [ ]:
proj = ccrs.PlateCarree()
fig = plt.figure(figsize=(fig_w, fig_h), dpi=dpi)
ax = fig.add_axes([0.045, 0.135, 0.94, 0.80], projection=proj)
ax.set_extent([lon_min, lon_max, lat_min, lat_max], crs=proj)
ax.add_feature(cfeature.LAND.with_scale(coast_res), facecolor="#ebe8e3", edgecolor="none", zorder=0)
ax.coastlines(resolution=coast_res, linewidth=0.5, zorder=1)
gl = ax.gridlines(draw_labels=True, linewidth=0.3, color="0.7", alpha=0.6, linestyle=":")
gl.top_labels = False; gl.right_labels = False

fig.legend(handles=[Patch(facecolor=c, label=n) for c, n in zip(MONTH_COLORS, MONTH_NAMES)],
           loc="lower center", bbox_to_anchor=(0.5, 0.0), ncol=12, fontsize=9, frameon=False,
           handlelength=1.3, columnspacing=1.2, title="release month", title_fontsize=9)
datetxt = fig.text(0.045, 0.952, "", fontsize=15, ha="left", va="bottom")
fig.text(0.985, 0.955,
         "%d releases x %d particles = %.2fM trajectories (of ~10061 per release)"
         % (len(releases), n_particles, len(releases) * n_particles / 1e6),
         ha="right", va="bottom", fontsize=9, color="0.35")

fig.canvas.draw()
buf = np.asarray(fig.canvas.buffer_rgba())      # WRITABLE view of the Agg framebuffer
H, W = buf.shape[:2]
assert W % 2 == 0 and H % 2 == 0, "libx264 yuv420p needs even frame dimensions: %dx%d" % (W, H)
print("canvas: %d x %d px, buffer writable: %s" % (W, H, buf.flags.writeable))

head = ax.scatter([], [], s=head_size, marker="o", linewidths=0, transform=proj, zorder=15)

## 5. Render (cumulative canvas -> ffmpeg pipe)

In [ ]:
cmd = [ffmpeg_path, "-y", "-loglevel", "error",
       "-f", "rawvideo", "-pix_fmt", "rgba", "-s", "%dx%d" % (W, H), "-r", str(fps), "-i", "-",
       "-an", "-c:v", "libx264", "-pix_fmt", "yuv420p",
       "-preset", preset, "-crf", str(crf), "-movflags", "+faststart", out_mp4]
print(" ".join(cmd[:1]), "...", out_mp4)
proc = subprocess.Popen(cmd, stdin=subprocess.PIPE)

rgba_trail = {m: np.array(mcolors.to_rgba(c, trail_alpha), dtype=np.float32)
              for m, c in enumerate(MONTH_COLORS, start=1)}
rgba_head  = {m: np.array(mcolors.to_rgba(c, 1.0), dtype=np.float32)
              for m, c in enumerate(MONTH_COLORS, start=1)}

t_render = time.time()
per_frame = np.zeros(n_frames, dtype=np.float32)
try:
    for fi in range(n_frames):
        tf = time.time()
        tnow = frame_times[fi]
        segs, cols, hx, hy, hc = [], [], [], [], []
        nlive = 0
        for r in releases:
            j = int((tnow - r["t0"]) / day)
            if j < 0 or j >= r["nv"]:
                continue
            a = max(0, j - frame_step_days)          # segments a..j are the NEW ones this frame
            if j > a:
                x = r["lon"][:, a:j + 1]; y = r["lat"][:, a:j + 1]
                x0 = x[:, :-1]; x1 = x[:, 1:]; y0 = y[:, :-1]; y1 = y[:, 1:]
                mm = np.isfinite(x0) & np.isfinite(x1) & np.isfinite(y0) & np.isfinite(y1)
                n = int(mm.sum())
                if n:
                    s4 = np.empty((n, 2, 2), dtype=np.float32)
                    s4[:, 0, 0] = x0[mm]; s4[:, 0, 1] = y0[mm]
                    s4[:, 1, 0] = x1[mm]; s4[:, 1, 1] = y1[mm]
                    segs.append(s4)
                    cols.append(np.repeat(rgba_trail[r["month"]][None, :], n, axis=0))
            m = np.isfinite(r["lon"][:, j]) & np.isfinite(r["lat"][:, j])
            nh = int(m.sum())
            if nh:
                hx.append(r["lon"][m, j]); hy.append(r["lat"][m, j])
                hc.append(np.repeat(rgba_head[r["month"]][None, :], nh, axis=0))
                nlive += nh

        # (a) paint the new trail segments PERMANENTLY into the framebuffer
        if segs:
            lc = LineCollection(np.concatenate(segs, axis=0),
                                colors=np.concatenate(cols, axis=0),
                                linewidths=trail_lw, transform=proj, zorder=2)
            ax.add_collection(lc, autolim=False)
            lc.set_clip_path(ax.patch)
            ax.draw_artist(lc)
            lc.remove()                      # pixels stay, artist is gone -> never redrawn

        snap = buf.copy()                    # trails-only state

        # (b) transient layer: live dots + date label
        if hx:
            head.set_offsets(np.column_stack([np.concatenate(hx), np.concatenate(hy)]))
            head.set_facecolor(np.concatenate(hc, axis=0))
        else:
            head.set_offsets(np.empty((0, 2), dtype=np.float32))
        datetxt.set_text("%s     live particles: %d"
                         % (np.datetime_as_string(tnow, unit="D"), nlive))
        ax.draw_artist(head)
        fig.draw_artist(datetxt)

        proc.stdin.write(buf.tobytes())      # (c) emit the frame
        buf[:] = snap                        # (d) roll back the transient layer

        per_frame[fi] = time.time() - tf
        if fi % 100 == 0 or fi == n_frames - 1:
            el = time.time() - t_render
            print("  frame %d/%d  %s  live=%d  %.0fs (%.3f s/frame, eta %.0f min)"
                  % (fi + 1, n_frames, np.datetime_as_string(tnow, unit="D"), nlive, el,
                     el / (fi + 1), (n_frames - fi - 1) * el / (fi + 1) / 60), flush=True)
finally:
    proc.stdin.close()
    rc = proc.wait()
render_s = time.time() - t_render
print("ffmpeg rc =", rc)
print("render finished in %.0f s (%.1f min), %.3f s/frame mean"
      % (render_s, render_s / 60, render_s / n_frames))
plt.close(fig)

## 6. Summary and verification

In [ ]:
size_mb = os.path.getsize(out_mp4) / 1e6
print("file           :", out_mp4)
print("size           : %.1f MB" % size_mb)
print("resolution     : %d x %d" % (W, H))
print("frames         : %d at %d fps -> %.1f s (%.1f min)"
      % (n_frames, fps, n_frames / fps, n_frames / fps / 60))
print("period         :", period_start, "..", period_end)
print("releases drawn :", len(releases), "(every %d days)" % release_spacing_days)
print("particles/rel  :", n_particles, "-> %d trajectories drawn" % (len(releases) * n_particles))
print("frame step     : %d days ; vertices: %d h" % (frame_step_days, obs_stride * 6))
print("load time      : %.0f s" % load_s)
print("render time    : %.0f s   (mean %.3f, median %.3f, max %.3f s/frame)"
      % (render_s, per_frame.mean(), np.median(per_frame), per_frame.max()))

probe = subprocess.run([ffmpeg_path, "-hide_banner", "-i", out_mp4],
                       capture_output=True, text=True)
print("\n--- ffmpeg -i ---")
print(probe.stderr.strip())